In [ ]:
# =========================================================
# CELL 1 — Setup
# =========================================================
import os, random, math
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_DIR = "/kaggle/input/global-wheat-detection"
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TRAIN_IMG_DIR = os.path.join(DATA_DIR, "train")
OUT_DIR = "/kaggle/working"

print("DATA_DIR:", DATA_DIR)
print("TRAIN_CSV exists:", os.path.exists(TRAIN_CSV))
print("TRAIN_IMG_DIR exists:", os.path.exists(TRAIN_IMG_DIR))
print("OUT_DIR:", OUT_DIR)

assert os.path.exists(TRAIN_CSV), "train.csv not found. Add competition data."
assert os.path.exists(TRAIN_IMG_DIR), "train images folder not found."

In [ ]:
# =========================================================
# CELL 2 — Load train.csv + bbox map
# =========================================================
import ast

df = pd.read_csv(TRAIN_CSV)
print("train.csv shape:", df.shape)
print(df.head())

def parse_bbox(bbox_str):
    return ast.literal_eval(bbox_str)  # [x,y,w,h]

df["bbox_list"] = df["bbox"].apply(parse_bbox)

bboxes_by_image = {}
size_by_image = {}

for row in df.itertuples(index=False):
    image_id = row.image_id
    x, y, w, h = row.bbox_list
    bboxes_by_image.setdefault(image_id, []).append([float(x), float(y), float(w), float(h)])
    size_by_image[image_id] = (int(row.width), int(row.height))

print("Unique images:", len(bboxes_by_image))
ex_id = next(iter(bboxes_by_image.keys()))
print("Example:", ex_id, "bbox_count:", len(bboxes_by_image[ex_id]), "size:", size_by_image[ex_id])


In [ ]:
# =========================================================
# CELL 2.5 — SAVE Detection BBox Overlay Samples (evidence)
# =========================================================
import matplotlib.pyplot as plt

try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False
    from PIL import Image, ImageDraw

N_SAMPLES = 20
MAX_BOXES = 80
SAVE_DIR = "/kaggle/working/dataset_preprocessed_s3/detection_overlay"
os.makedirs(SAVE_DIR, exist_ok=True)

all_ids = list(bboxes_by_image.keys())
random.shuffle(all_ids)
chosen_ids = all_ids[:N_SAMPLES]

def draw_and_save_overlay(image_id):
    img_path = os.path.join(TRAIN_IMG_DIR, f"{image_id}.jpg")
    if not os.path.exists(img_path):
        return None

    bboxes = bboxes_by_image.get(image_id, [])
    out_path = os.path.join(SAVE_DIR, f"bbox_overlay_{image_id}.png")

    if HAS_CV2:
        img = cv2.imread(img_path)
        if img is None:
            return None
        for (x, y, w, h) in bboxes[:MAX_BOXES]:
            x, y, w, h = int(x), int(y), int(w), int(h)
            cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 255), 2)
        cv2.imwrite(out_path, img)
        return out_path, cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = Image.open(img_path).convert("RGB")
        draw = ImageDraw.Draw(img)
        for (x, y, w, h) in bboxes[:MAX_BOXES]:
            x, y, w, h = int(x), int(y), int(w), int(h)
            draw.rectangle([x, y, x+w, y+h], outline=(255, 255, 0), width=3)
        img.save(out_path)
        return out_path, img

saved = []
for i, image_id in enumerate(chosen_ids):
    res = draw_and_save_overlay(image_id)
    if res is None:
        continue
    out_path, preview = res
    saved.append(out_path)
    if i < 3:
        plt.figure(figsize=(9,9))
        plt.imshow(preview)
        plt.title(f"BBox Overlay — {image_id}")
        plt.axis("off")
        plt.show()

print("Saved overlays:", len(saved))
print("Overlay folder:", SAVE_DIR)


In [ ]:
# =========================================================
# CELL 3 — Build samples (Positive/Negative crops)
# =========================================================
from tqdm.auto import tqdm

def iou_xywh(a, b):
    ax1, ay1, aw, ah = a
    bx1, by1, bw, bh = b
    ax2, ay2 = ax1 + aw, ay1 + ah
    bx2, by2 = bx1 + bw, by1 + bh
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    union = (aw*ah) + (bw*bh) - inter
    return inter/union if union > 0 else 0.0

def make_negative_box(img_w, img_h, pos_boxes, ref_box, max_tries=80, iou_thresh=0.05):
    _, _, w, h = ref_box
    w = int(max(8, min(float(w), float(img_w))))
    h = int(max(8, min(float(h), float(img_h))))
    if w >= img_w or h >= img_h:
        return None
    for _ in range(max_tries):
        x = random.randint(0, img_w - w)
        y = random.randint(0, img_h - h)
        cand = [float(x), float(y), float(w), float(h)]
        ok = True
        for pb in pos_boxes:
            if iou_xywh(cand, pb) >= iou_thresh:
                ok = False
                break
        if ok:
            return cand
    return None

NEG_PER_POS = 1
IOU_THRESH_NEG = 0.05

samples = []  # (img_path, box_xyxy_pixels, label)
pos_count, neg_count = 0, 0

items = list(bboxes_by_image.items())
for image_id, pos_boxes in tqdm(items, total=len(items), desc="Building crop samples"):
    img_path = os.path.join(TRAIN_IMG_DIR, f"{image_id}.jpg")
    if not os.path.exists(img_path):
        continue
    img_w, img_h = size_by_image[image_id]

    for pb in pos_boxes:
        x, y, w, h = pb
        x1 = max(0.0, float(x))
        y1 = max(0.0, float(y))
        x2 = min(float(img_w), x1 + float(w))
        y2 = min(float(img_h), y1 + float(h))
        if (x2-x1) < 2 or (y2-y1) < 2:
            continue

        samples.append((img_path, (x1,y1,x2,y2), 1))
        pos_count += 1

        for _ in range(NEG_PER_POS):
            nb = make_negative_box(img_w, img_h, pos_boxes, pb, iou_thresh=IOU_THRESH_NEG)
            if nb is None:
                continue
            nx, ny, nw, nh = nb
            samples.append((img_path, (nx, ny, nx+nw, ny+nh), 0))
            neg_count += 1

print("Positive:", pos_count, "| Negative:", neg_count, "| Total:", len(samples))


In [ ]:
# =========================================================
# CELL 4 — Split
# =========================================================
from sklearn.model_selection import train_test_split

paths = np.array([s[0] for s in samples])
boxes = np.array([s[1] for s in samples], dtype=np.float32)  # x1,y1,x2,y2 pixel
labels = np.array([s[2] for s in samples], dtype=np.int32)

X_train, X_tmp, b_train, b_tmp, y_train, y_tmp = train_test_split(
    paths, boxes, labels, test_size=0.30, random_state=SEED, stratify=labels
)
X_val, X_test, b_val, b_test, y_val, y_test = train_test_split(
    X_tmp, b_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=y_tmp
)

print("Train:", len(X_train), "Val:", len(X_val), "Test:", len(X_test))
print("Train pos%:", y_train.mean(), "Val pos%:", y_val.mean(), "Test pos%:", y_test.mean())


In [ ]:
# =========================================================
# CELL 5 — Install RemBG (Zero-shot segmentation, inference only)
# =========================================================
!pip -q install rembg onnxruntime pillow tqdm

from rembg import remove, new_session
from PIL import Image
import io

# pilih model yang lebih ringan agar cepat (u2netp)
REMBG_MODEL = "u2netp"
session = new_session(REMBG_MODEL)
print("RemBG session loaded:", REMBG_MODEL)

IMG_SIZE = 160

In [ ]:
# =========================================================
# CELL 5.5 — Generate preprocessing output folders (Raw crop vs RemBG crop)
# =========================================================
from tqdm.auto import tqdm

SAVE_ROOT = "/kaggle/working/dataset_preprocessed_s3"
RAW_DIR   = os.path.join(SAVE_ROOT, "raw_crops")
AI_DIR    = os.path.join(SAVE_ROOT, "rembg_crops")  # hasil zero-shot
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(AI_DIR,  exist_ok=True)

def pil_crop(img_pil, box_xyxy):
    x1,y1,x2,y2 = box_xyxy
    return img_pil.crop((int(x1), int(y1), int(x2), int(y2)))

def tight_crop_from_alpha(rgba_img, alpha_thresh=10):
    # rgba_img: PIL Image RGBA
    arr = np.array(rgba_img)
    alpha = arr[..., 3]
    ys, xs = np.where(alpha > alpha_thresh)
    if len(xs) == 0 or len(ys) == 0:
        return rgba_img  # fallback
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    # +1 so right/bottom included
    return rgba_img.crop((x1, y1, x2+1, y2+1))

def rembg_process_crop(crop_rgb_pil):
    # input: PIL RGB crop
    buf = io.BytesIO()
    crop_rgb_pil.save(buf, format="PNG")
    inp = buf.getvalue()

    out_bytes = remove(inp, session=session)  # returns PNG bytes, usually with alpha
    out_img = Image.open(io.BytesIO(out_bytes)).convert("RGBA")

    # auto-crop from alpha mask (crop otomatis dari model AI)
    out_img = tight_crop_from_alpha(out_img)

    # composite to RGB (black bg) for CNN
    bg = Image.new("RGB", out_img.size, (0,0,0))
    bg.paste(out_img, mask=out_img.split()[-1])  # alpha
    return bg

def save_preprocessed_set(X_paths, X_boxes, y, split_name, limit=None):
    """
    Save:
      raw crop -> RAW_DIR/split/label/
      rembg crop -> AI_DIR/split/label/
    """
    split_raw = os.path.join(RAW_DIR, split_name)
    split_ai  = os.path.join(AI_DIR, split_name)
    for lab in [0,1]:
        os.makedirs(os.path.join(split_raw, str(lab)), exist_ok=True)
        os.makedirs(os.path.join(split_ai,  str(lab)), exist_ok=True)

    idxs = np.arange(len(X_paths))
    if limit is not None:
        idxs = idxs[:limit]

    out_paths = []  # final AI paths for training
    out_labels = []

    for i in tqdm(idxs, desc=f"Preprocessing {split_name} (RemBG)"):
        img_path = X_paths[i]
        box = X_boxes[i]
        lab = int(y[i])

        img = Image.open(img_path).convert("RGB")
        raw_crop = pil_crop(img, box)

        # resize raw for saving (optional)
        raw_small = raw_crop.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        # rembg (zero-shot)
        ai_crop = rembg_process_crop(raw_crop)
        ai_small = ai_crop.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        base = os.path.splitext(os.path.basename(img_path))[0]
        fn = f"{base}_{i}_{lab}.png"

        raw_out = os.path.join(split_raw, str(lab), fn)
        ai_out  = os.path.join(split_ai,  str(lab), fn)

        raw_small.save(raw_out)
        ai_small.save(ai_out)

        out_paths.append(ai_out)
        out_labels.append(lab)

    return np.array(out_paths), np.array(out_labels, dtype=np.int32)

# OPTIONAL: kalau takut lama, batasi dulu untuk uji cepat
# TRAIN_LIMIT = 30000
# VAL_LIMIT   = 6000
# TEST_LIMIT  = 6000
TRAIN_LIMIT = 600
VAL_LIMIT   = 300
TEST_LIMIT  = 300

train_ai_paths, train_ai_labels = save_preprocessed_set(X_train, b_train, y_train, "train", limit=TRAIN_LIMIT)
val_ai_paths,   val_ai_labels   = save_preprocessed_set(X_val,   b_val,   y_val,   "val",   limit=VAL_LIMIT)
test_ai_paths,  test_ai_labels  = save_preprocessed_set(X_test,  b_test,  y_test,  "test",  limit=TEST_LIMIT)

print("Saved preprocessing folder:", SAVE_ROOT)
print("Example RemBG file:", train_ai_paths[0] if len(train_ai_paths)>0 else None)

In [ ]:
# =========================================================
# CELL 5.6 — Preview preprocessing results
# =========================================================
import matplotlib.pyplot as plt

def show_pair(split="train", lab=1, n=3):
    raw_dir = os.path.join(RAW_DIR, split, str(lab))
    ai_dir  = os.path.join(AI_DIR,  split, str(lab))
    fns = sorted(os.listdir(ai_dir))[:n]
    for fn in fns:
        raw_img = Image.open(os.path.join(raw_dir, fn)).convert("RGB")
        ai_img  = Image.open(os.path.join(ai_dir, fn)).convert("RGB")

        plt.figure(figsize=(8,4))
        plt.subplot(1,2,1); plt.imshow(raw_img); plt.title("Raw crop"); plt.axis("off")
        plt.subplot(1,2,2); plt.imshow(ai_img);  plt.title("RemBG (zero-shot)"); plt.axis("off")
        plt.tight_layout()
        plt.show()

show_pair(split="train", lab=1, n=3)
show_pair(split="train", lab=0, n=3)


In [ ]:
# =========================================================
# CELL 6 — TF Dataset
# =========================================================
import tensorflow as tf

BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

def load_png(path, label):
    img_bytes = tf.io.read_file(path)
    img = tf.image.decode_png(img_bytes, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)  # [0..1]
    return img, tf.cast(label, tf.float32)

def make_ds(paths, labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(load_png, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

train_ds = make_ds(train_ai_paths, train_ai_labels, training=True)
val_ds   = make_ds(val_ai_paths,   val_ai_labels,   training=False)
test_ds  = make_ds(test_ai_paths,  test_ai_labels,  training=False)

xb, yb = next(iter(train_ds))
print("Batch:", xb.shape, yb.shape)


In [ ]:
# =========================================================
# CELL 7 — Simple CNN
# =========================================================
from tensorflow.keras import layers, models

def build_simple_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3)):
    inputs = layers.Input(shape=input_shape)
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inputs)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inputs, outputs)

model = build_simple_cnn()
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="acc", threshold=0.5),
        tf.keras.metrics.Precision(name="precision", thresholds=0.5),
        tf.keras.metrics.Recall(name="recall", thresholds=0.5),
    ],
)

model.summary()


In [ ]:
# =========================================================
# CELL 8 — Train (EPOCHS fixed = 2)
# =========================================================
EPOCHS = 2
ckpt_path = os.path.join(OUT_DIR, "cnn_rembg_best.keras")

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, verbose=0),
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True, verbose=1),
]

try:
    from tqdm.keras import TqdmCallback
    callbacks.append(TqdmCallback(verbose=1))
    fit_verbose = 0
except Exception:
    fit_verbose = 1

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=fit_verbose
)

print("Saved best model:", ckpt_path)


In [ ]:
# =========================================================
# CELL 9 — Evaluate
# =========================================================
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

best_model = tf.keras.models.load_model(ckpt_path)

y_prob = best_model.predict(test_ds, verbose=1).reshape(-1)
y_pred = (y_prob >= 0.5).astype(int)
y_true = test_ai_labels.astype(int)

acc  = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec  = recall_score(y_true, y_pred, zero_division=0)
f1   = f1_score(y_true, y_pred, zero_division=0)
cm   = confusion_matrix(y_true, y_pred)

tn, fp, fn, tp = cm.ravel()
spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0

print("=== TEST METRICS (Scenario 3: RemBG Zero-shot + Simple CNN) ===")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"Specificity: {spec:.4f}")
print(f"F1-Score : {f1:.4f}")
print("\nConfusion Matrix [[TN FP],[FN TP]]:\n", cm)


In [ ]:
# =========================================================
# CELL 10 — Visualizations
# =========================================================
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(5,4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xticks([0,1], ["Pred 0", "Pred 1"])
plt.yticks([0,1], ["True 0", "True 1"])
for (i,j), val in np.ndenumerate(cm):
    plt.text(j, i, int(val), ha="center", va="center")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

hist = history.history
plt.figure(figsize=(6,4))
plt.plot(hist.get("loss", []), label="train_loss")
plt.plot(hist.get("val_loss", []), label="val_loss")
plt.title("Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()

plt.figure(figsize=(6,4))
plt.plot(hist.get("acc", []), label="train_acc")
plt.plot(hist.get("val_acc", []), label="val_acc")
plt.title("Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()


In [ ]:
# =========================================================
# CELL 11 — Save metrics
# =========================================================
import pandas as pd

metrics_df = pd.DataFrame([{
    "scenario": "ZeroShot_RemBG_u2netp",
    "model": "SimpleCNN",
    "img_size": IMG_SIZE,
    "neg_per_pos": NEG_PER_POS,
    "iou_thresh_neg": IOU_THRESH_NEG,
    "epochs": EPOCHS,
    "accuracy": acc,
    "precision": prec,
    "recall": rec,
    "specificity": spec,
    "f1_score": f1,
    "seed": SEED
}])

csv_path = os.path.join(OUT_DIR, "metrics_s3_rembg_simplecnn.csv")
metrics_df.to_csv(csv_path, index=False)

print("Saved:", csv_path)
metrics_df
